# Study 937 — Tranches — the teardown

The timing-luck cone by tranche count, the joint block-bootstrap CIs, the (selection-biased) best-minus-worst *t*, the persistence test, the era cut, the cost and ticket sweeps, two cross-checks and the live synthetic control.

**Design.** One rule (monthly 200-day trend, SPY vs IEF), 21 rebalance offsets, the state known at the close of *t* earning the return of *t+1* (one shift — so execution sits at the *t* close; a second day of delay is re-run in results.md and moves nothing), cost = one-way bps x traded notional (2 x |Δpos| on a switch), no shorts so no borrow, Sharpes excess-of-cash (^IRX accrual PROXY; BIL cross-check). An N-tranche book holds N sleeves on offsets floor(k·21/N), each self-financing; the book return is their NAV-share weighted average, evaluated at all 21 rotations of the anchor.

Every real number is frozen from `docs/results.md` (Fingerprint `e8f9552be0ac`, as-of 2026-06-30); the live cells are synthetic and labelled.

In [1]:
R = {'tape_start': '2002-07-30', 'start': '2003-06-16', 'end': '2026-06-30', 'n_tape': 6018, 'n_days': 5797, 'fp': 'e8f9552be0ac', 'in_risky': 81.0, 'n1_sharpe': 0.631, 'n1_sd': 0.066, 'n1_min': 0.499, 'n1_max': 0.755, 'n1_range': 0.256, 'n1_cagr_sd': 0.86, 'n1_tw': 95.1, 'n1_turn': 2.741, 'n4_sharpe': 0.67, 'n4_sd': 0.023, 'n4_range': 0.092, 'n4_tw': 27.3, 'n4_turn': 2.71, 'n12_sharpe': 0.675, 'n12_sd': 0.01, 'n12_range': 0.034, 'n12_tw': 7.7, 'n12_turn': 2.7, 'n21_sharpe': 0.676, 'n21_sd': 0.0, 'n21_range': 0.0, 'n21_tw': 0.0, 'n21_turn': 2.7, 'shrink4': 0.34, 'shrink12': 0.152, 'shrink21': 0.0, 'gain': 0.044, 'cagr_1': 9.63, 'cagr_21': 9.7, 'cagr_gain': 0.07, 'vol_1': 13.3, 'vol_21': 12.32, 'dd_1': -31.0, 'dd_1_worst': -35.4, 'dd_21': -26.2, 'turn_change': -1.5, 'sd_ci_lo': 0.054, 'sd_ci_hi': 0.145, 'gain_ci_lo': 0.014, 'gain_ci_hi': 0.073, 'gain_neg': 0.3, 'best_off': 2, 'worst_off': 17, 'best_sh': 0.755, 'worst_sh': 0.499, 'bw_gap': 0.256, 'bw_t': 1.98, 'rho_eras': 0.309, 'split': '2014-12-17', 'early_winner': 2, 'late_rank': 2, 'late_sh_winner': 0.62, 'late_sh_mean': 0.562, 'late_sh_best': 0.635, 'late_winner': 9, 'rho_rules': 0.556, 'chase_late': 0.058, 'gain_late': 0.042, 'gain_early': 0.044, 'rnd_sharpe': 0.515, 'rnd_sd': 0.103, 'rnd_range': 0.462, 'rnd_tw': 419.0, 'rnd_tranched': 0.59, 'rnd_seeds': 20, 'rnd_sd_mean': 0.093, 'rnd_sd_med': 0.094, 'rnd_sd_min': 0.065, 'rnd_sd_max': 0.121, 'rnd_n_wider': 19, 'lag_sh': 0.622, 'lag_sd': 0.071, 'lag_range': 0.27, 'lag_full': 0.666, 'lag_gain': 0.044, 'n4_min': 0.634, 'n4_max': 0.726, 'era_e_n': 2656, 'era_e_sh': 0.717, 'era_e_sd': 0.096, 'era_e_range': 0.341, 'era_e_full': 0.762, 'era_e_gain': 0.045, 'era_l_n': 3119, 'era_l_sh': 0.604, 'era_l_sd': 0.056, 'era_l_range': 0.218, 'era_l_full': 0.645, 'era_l_gain': 0.041, 'cost0_gain': 0.045, 'cost0_sd': 0.065, 'cost25_gain': 0.042, 'cost25_sd': 0.071, 'tick1': 2.6, 'tick4': 11.0, 'tick12': 32.3, 'tick21': 57.6, 'drag21_half': 0.288, 'drag21_two': 1.151, 'drag4_half': 0.055, 'mom_sd': 0.039, 'mom_range': 0.165, 'mom_tw': 77.7, 'mom_gain': 0.013, 'mom_turn_1': 2.81, 'mom_turn_21': 2.82, 'bil_n': 4581, 'bil_sd': 0.051, 'bil_range': 0.206, 'bil_gain': 0.046, 'bil_proxy_diff': 0.008, 'syn_pl_edge': 0.45, 'syn_pl_min': 0.249, 'syn_pl_sd1': 0.048, 'syn_pl_gain': 0.024, 'syn_nl_edge': -0.003, 'syn_nl_max': 0.175, 'syn_nl_sd1': 0.053, 'syn_nl_gain': 0.005}

## 1. The cone by tranche count

Dispersion is measured across the 21 rotations of each N-tranche book, so every row answers the same question: *how much does the arbitrary anchor date matter?*

In [2]:
print(f"n = {R['n_days']:,} days, {R['start']} -> {R['end']}, in-risky {R['in_risky']:.1f}%")
print()
print(' N   Sharpe    sd      min     max    range   CAGRsd   TWspread  traded/yr')
rows = [(1, R['n1_sharpe'], R['n1_sd'], R['n1_min'], R['n1_max'], R['n1_range'], R['n1_cagr_sd'], R['n1_tw'], R['n1_turn']),
        (4, R['n4_sharpe'], R['n4_sd'], 0.634, 0.726, R['n4_range'], 0.29, R['n4_tw'], R['n4_turn']),
        (12, R['n12_sharpe'], R['n12_sd'], 0.662, 0.696, R['n12_range'], 0.10, R['n12_tw'], R['n12_turn']),
        (21, R['n21_sharpe'], R['n21_sd'], 0.676, 0.676, R['n21_range'], 0.00, R['n21_tw'], R['n21_turn'])]
for n, sh, sd, lo, hi, rg, cs, tw, tn in rows:
    print(f'{n:2d}   {sh:+.3f}   {sd:.3f}  {lo:+.3f}  {hi:+.3f}   {rg:.3f}   {cs:4.2f}pp  {tw:6.1f}%   {tn:.3f}x')
print()
print(f"shrink sd(N)/sd(1): 1.000 / {R['shrink4']:.3f} / {R['shrink12']:.3f} / {R['shrink21']:.3f}")
print(' 1/sqrt(N) reference: 1.000 / 0.500 / 0.289 / 0.218  -> the collapse beats it')

n = 5,797 days, 2003-06-16 -> 2026-06-30, in-risky 81.0%

 N   Sharpe    sd      min     max    range   CAGRsd   TWspread  traded/yr
 1   +0.631   0.066  +0.499  +0.755   0.256   0.86pp    95.1%   2.741x
 4   +0.670   0.023  +0.634  +0.726   0.092   0.29pp    27.3%   2.710x
12   +0.675   0.010  +0.662  +0.696   0.034   0.10pp     7.7%   2.700x
21   +0.676   0.000  +0.676  +0.676   0.000   0.00pp     0.0%   2.700x

shrink sd(N)/sd(1): 1.000 / 0.340 / 0.152 / 0.000
 1/sqrt(N) reference: 1.000 / 0.500 / 0.289 / 0.218  -> the collapse beats it


> 💡 **In plain words.** Averaging 21 highly-correlated books cancels the part of each that is pure sampling phase. It beats 1/√N because the residuals are not independent draws — they are the *same* signal sampled at different phases, and the full set of phases reconstructs the underlying rule exactly.

## 2. What the gain is made of

The 21-tranche book **is** the NAV-weighted average of the 21 single-date books, so its mean return is theirs by construction. Any Sharpe gain must therefore come from the denominator — and it does.

In [3]:
print(f"Sharpe {R['n1_sharpe']:+.3f} -> {R['n21_sharpe']:+.3f}   gain {R['gain']:+.3f}")
print(f"CAGR   {R['cagr_1']:.2f}% -> {R['cagr_21']:.2f}%   ({R['cagr_gain']:+.2f} pp/yr)")
print(f"vol    {R['vol_1']:.2f}% -> {R['vol_21']:.2f}%   ({R['vol_21']-R['vol_1']:+.2f} pp)")
print(f"maxDD  mean {R['dd_1']:.1f}% (worst placement {R['dd_1_worst']:.1f}%) -> {R['dd_21']:.1f}%")
print()
m = R['n1_sharpe'] * R['vol_1']
print(f"implied mean excess return, one date: {m:.2f}%/yr")
print(f"hold that numerator fixed, apply the tranched vol: {m/R['vol_21']:+.3f} "
      f"vs the actual {R['n21_sharpe']:+.3f}")
print('-> the gain is the denominator. The numerator cannot move: the tranched book')
print('   IS the NAV-weighted average of the 21 single-date books.')

Sharpe +0.631 -> +0.676   gain +0.044
CAGR   9.63% -> 9.70%   (+0.07 pp/yr)
vol    13.30% -> 12.32%   (-0.98 pp)
maxDD  mean -31.0% (worst placement -35.4%) -> -26.2%

implied mean excess return, one date: 8.39%/yr
hold that numerator fixed, apply the tranched vol: +0.681 vs the actual +0.676
-> the gain is the denominator. The numerator cannot move: the tranched book
   IS the NAV-weighted average of the 21 single-date books.


## 3. Joint block bootstrap (1,000 draws, 21-day blocks)

Rows are resampled **jointly** across the 21 books plus the tranched book, so the cross-sectional dependence that makes the cone narrow-ish survives the resample.

**One of these two rows is inference and the other is decoration.** A standard deviation is non-negative, so an interval on the *cone sd* excludes zero for any 21 non-identical books — it tests nothing. It is also badly right-skewed, the point sitting on its lower edge, because resampling blocks of days scrambles the very calendar phases that generate the cone. The **gain** is the row that could have printed negative and did not.

In [4]:
print(f"cone sd at N=1 : {R['n1_sd']:.3f}  95% band [{R['sd_ci_lo']:.3f}, {R['sd_ci_hi']:.3f}]")
print('                 ^ a SPREAD BAND, not a test: an sd cannot be negative.')
print(f"tranching gain : {R['gain']:+.3f}  95% CI [{R['gain_ci_lo']:+.3f}, {R['gain_ci_hi']:+.3f}]  "
      f"share<0 {R['gain_neg']:.1f}%")
from statistics import NormalDist
print(f"                 ^ Gaussian equivalent |t| = {NormalDist().inv_cdf(1 - R['gain_neg']/100):.2f} "
      f"(NOT 2.9 — that was a rounding of the wrong tail), on a quantity that is")
print('                   near-mechanical anyway: averaging imperfectly correlated')
print('                   books at a fixed mean return almost has to cut the vol.')

cone sd at N=1 : 0.066  95% band [0.054, 0.145]
                 ^ a SPREAD BAND, not a test: an sd cannot be negative.
tranching gain : +0.044  95% CI [+0.014, +0.073]  share<0 0.3%
                 ^ Gaussian equivalent |t| = 2.75 (NOT 2.9 — that was a rounding of the wrong tail), on a quantity that is
                   near-mechanical anyway: averaging imperfectly correlated
                   books at a fixed mean return almost has to cut the vol.


## 4. Is the lucky offset forecastable? (it must not be)

The best-minus-worst *t* is reported **only** as an upper bound: it compares the argmax with the argmin of 21 correlated books, so it is selection-biased by construction and cannot be read as evidence of a tradable spread.

In [5]:
print(f"ex-post best offset {R['best_off']} ({R['best_sh']:+.3f}) vs worst {R['worst_off']} ({R['worst_sh']:+.3f})")
print(f"  gap {R['bw_gap']:.3f}, HAC t on the daily return difference {R['bw_t']:+.2f}  <- SELECTION-BIASED")
print(f"persistence (split {R['split']}): rank correlation across halves rho = {R['rho_eras']:+.3f}")
print(f"  first-half winner (offset {R['early_winner']}) ranked {R['late_rank']}/21 later: "
      f"{R['late_sh_winner']:+.3f} vs the 21-date mean {R['late_sh_mean']:+.3f} "
      f"(+{R['chase_late']:.3f}), while the true late winner was offset {R['late_winner']}")
print(f"  SAME-WINDOW: chasing it earned {R['chase_late']:+.3f} in the second half vs "
      f"{R['gain_late']:+.3f} for tranching over the SAME rows -> the chase WINS by "
      f"{R['chase_late']-R['gain_late']:+.3f}.")
print('  It is one ex-post-selected draw, not an edge: the late winner was a different')
print('  offset, rho is only +0.309, and the chaser re-enters the lottery he just won.')
print('  (Comparing it against the FULL-SAMPLE +0.044 would flatter us. It does not.)')
print(f"cross-rule offset-ranking correlation: rho = {R['rho_rules']:+.3f} "
      f"-> some shared month-end footprint, smaller than the cone it sits in")
print()
print(f"edge-free control (seed 937): exposure-matched RANDOM timing dispersal sd {R['rnd_sd']:.3f}, "
      f"range {R['rnd_range']:.3f}, terminal spread {R['rnd_tw']:.0f}% -> tranched {R['rnd_tranched']:+.3f}")
print(f"  over {R['rnd_seeds']} seeds: sd mean {R['rnd_sd_mean']:.3f}, median {R['rnd_sd_med']:.3f}, "
      f"min {R['rnd_sd_min']:.3f}, max {R['rnd_sd_max']:.3f}; wider than the trend rule in "
      f"{R['rnd_n_wider']}/{R['rnd_seeds']} (the quoted 0.103 is an upper-half draw)")
print('A rule with zero timing ability has a WIDER cone than the trend rule: artefact, not skill.')

ex-post best offset 2 (+0.755) vs worst 17 (+0.499)
  gap 0.256, HAC t on the daily return difference +1.98  <- SELECTION-BIASED
persistence (split 2014-12-17): rank correlation across halves rho = +0.309
  first-half winner (offset 2) ranked 2/21 later: +0.620 vs the 21-date mean +0.562 (+0.058), while the true late winner was offset 9
  SAME-WINDOW: chasing it earned +0.058 in the second half vs +0.042 for tranching over the SAME rows -> the chase WINS by +0.016.
  It is one ex-post-selected draw, not an edge: the late winner was a different
  offset, rho is only +0.309, and the chaser re-enters the lottery he just won.
  (Comparing it against the FULL-SAMPLE +0.044 would flatter us. It does not.)
cross-rule offset-ranking correlation: rho = +0.556 -> some shared month-end footprint, smaller than the cone it sits in

edge-free control (seed 937): exposure-matched RANDOM timing dispersal sd 0.103, range 0.462, terminal spread 419% -> tranched +0.590
  over 20 seeds: sd mean 0.093, med

## 5. Era cut, cost sweep, ticket assumption

Costs are charged one-way on traded notional; nothing is shorted, so no borrow leg exists. The **ticket** figures are a labelled ASSUMPTION — broker fees are not on the price tape — and they are the only cost that scales with N.

In [6]:
print(f"2003-2013 (n={R['era_e_n']}): one date {R['era_e_sh']:+.3f} (sd {R['era_e_sd']:.3f}, "
      f"range {R['era_e_range']:.3f}) -> tranched {R['era_e_full']:+.3f}  gain {R['era_e_gain']:+.3f}")
print(f"2014-2026 (n={R['era_l_n']}): one date {R['era_l_sh']:+.3f} (sd {R['era_l_sd']:.3f}, "
      f"range {R['era_l_range']:.3f}) -> tranched {R['era_l_full']:+.3f}  gain {R['era_l_gain']:+.3f}")
print()
print(f"cost 0 bps : gain {R['cost0_gain']:+.3f} (cone sd {R['cost0_sd']:.3f})")
print(f"cost 25 bps: gain {R['cost25_gain']:+.3f} (cone sd {R['cost25_sd']:.3f})  "
      f"-> friction does not drive this; the cone even widens, unlucky dates trade more")
print()
for n, tk in [(1, R['tick1']), (4, R['tick4']), (12, R['tick12']), (21, R['tick21'])]:
    print(f"  N={n:2d}: {tk:5.1f} tickets/yr -> {tk*0.5/1e4*100:.3f}%/yr at 0.5bp/ticket, "
          f"{tk*2.0/1e4*100:.3f}%/yr at 2bp/ticket")
print(f"the 21-tranche ticket bill at 0.5bp ({R['drag21_half']:.3f}%/yr) exceeds the "
      f"{R['cagr_gain']:.2f}pp/yr of CAGR it adds — size-dependent, and not modelled in the Sharpes above")

2003-2013 (n=2656): one date +0.717 (sd 0.096, range 0.341) -> tranched +0.762  gain +0.045
2014-2026 (n=3119): one date +0.604 (sd 0.056, range 0.218) -> tranched +0.645  gain +0.041

cost 0 bps : gain +0.045 (cone sd 0.065)
cost 25 bps: gain +0.042 (cone sd 0.071)  -> friction does not drive this; the cone even widens, unlucky dates trade more

  N= 1:   2.6 tickets/yr -> 0.013%/yr at 0.5bp/ticket, 0.052%/yr at 2bp/ticket
  N= 4:  11.0 tickets/yr -> 0.055%/yr at 0.5bp/ticket, 0.220%/yr at 2bp/ticket
  N=12:  32.3 tickets/yr -> 0.161%/yr at 0.5bp/ticket, 0.646%/yr at 2bp/ticket
  N=21:  57.6 tickets/yr -> 0.288%/yr at 0.5bp/ticket, 1.152%/yr at 2bp/ticket
the 21-tranche ticket bill at 0.5bp (0.288%/yr) exceeds the 0.07pp/yr of CAGR it adds — size-dependent, and not modelled in the Sharpes above


## 6. Cross-checks

In [7]:
print(f"12-1 momentum sleeve : cone sd {R['mom_sd']:.3f}, range {R['mom_range']:.3f}, "
      f"terminal spread {R['mom_tw']:.1f}% -> 0.000 tranched; gain {R['mom_gain']:+.3f}; "
      f"turnover {R['mom_turn_1']:.2f}x -> {R['mom_turn_21']:.2f}x")
print(f"tradable BIL cash leg: n={R['bil_n']}, cone sd {R['bil_sd']:.3f}, range {R['bil_range']:.3f}, "
      f"gain {R['bil_gain']:+.3f}; the ^IRX PROXY moves the Sharpe by {R['bil_proxy_diff']:.3f}")
print(f"one MORE day of delay: single-date {R['lag_sh']:+.3f} (sd {R['lag_sd']:.3f}, "
      f"range {R['lag_range']:.3f}) -> tranched {R['lag_full']:+.3f}, gain {R['lag_gain']:+.3f} "
      f"-> not an execution-timing artefact")
print(f"N=4 rotations span {R['n4_min']:+.3f} to {R['n4_max']:+.3f}: every one of the 21 "
      f"four-tranche placements beat the average single date ({R['n1_sharpe']:+.3f})")

12-1 momentum sleeve : cone sd 0.039, range 0.165, terminal spread 77.7% -> 0.000 tranched; gain +0.013; turnover 2.81x -> 2.82x
tradable BIL cash leg: n=4581, cone sd 0.051, range 0.206, gain +0.046; the ^IRX PROXY moves the Sharpe by 0.008
one MORE day of delay: single-date +0.622 (sd 0.071, range 0.270) -> tranched +0.666, gain +0.044 -> not an execution-timing artefact
N=4 rotations span +0.634 to +0.726: every one of the 21 four-tranche placements beat the average single date (+0.631)


## 7. Live synthetic control — planted vs null (offline, deterministic)

Positive control: with a planted regime the sleeve must beat an exposure-matched random control. Negative control: on the flat null it must not. In both worlds the cone must exist and tranching must collapse it — that is what makes the fix orthogonal to the edge.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from tranching import data, strategy as st
import numpy as np
for tag, ss in [('planted', 1.0), ('null   ', 0.0)]:
    rows = [st.synthetic_detect(p) for p, _ in
            data.synthetic_panel(3, signal_strength=ss, n_years=40)]
    edge = np.array([d['sleeve_minus_random'] for d in rows])
    print('%s (3 worlds): sleeve-vs-random %+.3f [%+.3f, %+.3f] | cone sd %.3f -> %.3f | gain %+.3f'
          % (tag, edge.mean(), edge.min(), edge.max(),
             np.mean([d['sharpe_sd_n1'] for d in rows]),
             np.mean([d['sharpe_sd_full'] for d in rows]),
             np.mean([d['gain'] for d in rows])))
print()
print('frozen 6-seed summary from docs/results.md: planted edge %+.3f (min %+.3f), '
      'null edge %+.3f (max %+.3f)' % (R['syn_pl_edge'], R['syn_pl_min'], R['syn_nl_edge'], R['syn_nl_max']))

planted (3 worlds): sleeve-vs-random +0.536 [+0.380, +0.743] | cone sd 0.047 -> 0.000 | gain +0.027


null    (3 worlds): sleeve-vs-random +0.044 [-0.170, +0.175] | cone sd 0.052 -> 0.000 | gain +0.007

frozen 6-seed summary from docs/results.md: planted edge +0.450 (min +0.249), null edge -0.003 (max +0.175)


## Verdict

- **Signal — Real.** The rebalance-date cone is measurable on the real tape at sd **0.066**, range **0.256** Sharpe and **95%** of terminal wealth; it survives an era cut (0.096 / 0.056), a second rule (0.039), the tradable cash leg (0.051), an extra day of execution delay (0.071), and is *wider* for an edge-free rule in 19/20 seeds (mean 0.093) — the signature of an artefact, which is exactly the claim. The stamp rests on **size and replication, not on a p-value**: a dispersion has no null to reject, and the sd's bootstrap band cannot straddle zero whatever the data say. Tranching removes it identically (sd 0.000 at N=21, 0.340 of it left at N=4) with a Sharpe gain of **+0.044** (CI [+0.014, +0.073], 0.3% of draws negative — the study's one real interval), stable across eras and from 0 to 25 bps. Forecasting the lucky date did beat the fix in the one out-of-sample half we have (+0.058 against +0.042 on the same rows, ρ = +0.309) — one ex-post-selected draw, reported as it fell, and not something we would size a book on.
- **Tradability — Fragile.** Proportional cost is a non-issue (2.741x → 2.700x NAV/yr), but the gain is variance, not return (**+0.07 pp/yr** of CAGR), and the unpriced ticket bill (2.6 → 57.6 tickets/yr; 0.288%/yr at 0.5 bp each) can exceed it outright on a small account. Tax, likewise, is not modelled and runs against high N. Four tranches is the defensible operating point: 66% of the cone gone for 11 tickets a year.